In [1]:
!uv sync

Resolved 17 packages in 22ms
Audited 16 packages in 4ms


In [ ]:
# Imports
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from mandrop.engine import make_step, compute_macros, init_state
from mandrop.generator import setup, boundary_stats
from mandrop.run import run
from mandrop.viz import plot_fields

print(f"JAX {jax.__version__}, devices: {jax.devices()}")

In [ ]:
# Resolution and domain
RESOLUTION_UM       = 2.5      # µm per lattice unit (1.0=finer, 2.5=faster)
OUTLET_EXTRA_MM     = 0.3575   # extend outlet 2×

# ── Physical chip operating point ────────────────────────────────────────
NU_PHYS      = 1e-6       # m²/s (water)
RHO_PHYS     = 1000.0     # kg/m³
U_OIL_PHYS   = 0.32       # m/s per oil inlet (150 µL/min ÷ 77.5×100 µm)
U_TOP_PHYS   = 0.053      # m/s central water (40 µL/min ÷ 125×100 µm)
U_SIDE_PHYS  = 0.043      # m/s per side water (20 µL/min ÷ 77.5×100 µm)
SIGMA_EQ_PHYS    = 5e-3   # N/m equilibrium IFT (HFE-7500 + 2% PicoSurf)
SIGMA_CLEAN_PHYS = 5e-3   # N/m bare IFT (= σ_eq → Stage 3 disabled)

# Numerical knob — inlet Mach cap; throat focusing ~3.6× → throat Mach ≈ 0.09
MACH_TARGET = 0.025

# Derived lattice values
dx        = RESOLUTION_UM * 1e-6
nu_lu     = (MACH_TARGET / (3**0.5)) / U_OIL_PHYS * NU_PHYS / dx
nu_lu     = max(min(nu_lu, 0.5), 0.02)
dt        = nu_lu * dx**2 / NU_PHYS
u_lu_per_mps   = dt / dx
sigma_lu_to_Nm = RHO_PHYS * dx**3 / dt**2
p_lu_to_pa     = RHO_PHYS * (dx/dt)**2

W           = 4.0
SIGMA_EQ    = SIGMA_EQ_PHYS    / sigma_lu_to_Nm
SIGMA_CLEAN = SIGMA_CLEAN_PHYS / sigma_lu_to_Nm
TAU_ADS_LU  = 2000.0
D_GAMMA     = 0.001
rho0  = 1.0
nu    = nu_lu
tau_f = 3.0 * nu + 0.5
M_ch  = 0.05

U_OIL_IN_LU        = U_OIL_PHYS  * u_lu_per_mps
U_TOP_IN_LU        = U_TOP_PHYS  * u_lu_per_mps
U_WATER_SIDE_IN_LU = U_SIDE_PHYS * u_lu_per_mps

drho  = 0.001
F_OUT = -1.0

print(f"dx={dx*1e6:.1f} µm  dt={dt*1e9:.1f} ns  ν_lu={nu_lu:.4f}  τ={tau_f:.3f}")
print(f"u_oil_lu={U_OIL_IN_LU:.4f}  inlet Mach={U_OIL_IN_LU*3**0.5:.3f}  σ_lu={SIGMA_EQ:.4f}")
print(f"1 lu of ρ ≈ {p_lu_to_pa/3:.0f} Pa  1 sec = {int(1/dt):,} lu_ts")

In [ ]:
# Geometry (rasterized from cropped.dxf) and step function
geo = setup(
    resolution_um      =RESOLUTION_UM,
    outlet_extra_mm    =OUTLET_EXTRA_MM,
    u_top_in_lu       =U_TOP_IN_LU,
    u_water_side_in_lu=U_WATER_SIDE_IN_LU,
    u_oil_in_lu       =U_OIL_IN_LU,
    rho_out           =rho0 + F_OUT * drho / 2.0,
)
p = geo["params"]
Nx, Ny = p["Nx"], p["Ny"]

step = make_step(
    geo["wall"], geo["fluid"], geo["interior"], geo["opp_jnp"],
    tau_f, SIGMA_CLEAN, SIGMA_EQ, W, TAU_ADS_LU, D_GAMMA, M_ch,
    geo["apply_f_bcs"], geo["apply_phi_bcs"], geo["apply_gamma_bcs"], geo["boundary_mask"],
)

print(f"Resolution: {p['resolution_um']} µm/lu  Domain: {Nx}×{Ny}")
print(f"Channel x∈[{p['gxL']},{p['gxR']}]  Throat x∈[{p['gxTL']},{p['gxTR']}]")
print(f"Inlets (lu/ts): top u_y=-{p['u_top_in_lu']:.4f}  water_side |u_x|={p['u_water_side_in_lu']:.4f}  oil |u_x|={p['u_oil_in_lu']:.4f}")
print(f"Outlet rho: {p['rho_out']:.4f}")
print(f"σ_clean={SIGMA_CLEAN}  σ_eq={SIGMA_EQ}  τ_ads={TAU_ADS_LU}  W={W}")

In [ ]:
# Initial state: equilibrium f + water prefill + relaxed phi profile + Γ on interface band
f0, phi0, Gamma0 = init_state(
    Nx, Ny, rho0, geo["apply_phi_bcs"], geo["apply_gamma_bcs"],
    SIGMA_EQ, W, M_ch, geo["water_prefill"],
)
interior = geo["interior"]

print(f"Water prefill: {int(geo['water_prefill'].sum())} pixels")
print(f"Initial water (φ<0.5): {int((phi0 < 0.5).sum())} pixels")
print(f"Γ=1 (aged interface band): {int((Gamma0 > 0.5).sum())} pixels")

In [ ]:
# Visualize geometry and initial phi distribution
fig, axes = plt.subplots(1, 2, figsize=(8, 12))

im0 = axes[0].imshow(geo["wall"].T, origin='lower', cmap='gray_r', vmin=0, vmax=1)
axes[0].set_title('Wall mask')
axes[0].axhline(p['Y_USLOT_TOP'], color='b', ls=':', alpha=0.5, label='upper slots (water)')
axes[0].axhline(p['Y_USLOT_BOT'], color='b', ls=':', alpha=0.5)
axes[0].axhline(p['Y_LSLOT_TOP'], color='r', ls=':', alpha=0.5, label='lower slots (oil)')
axes[0].axhline(p['Y_LSLOT_BOT'], color='r', ls=':', alpha=0.5)
axes[0].legend(loc='upper right', fontsize=8)
plt.colorbar(im0, ax=axes[0], shrink=0.5)

im1 = axes[1].imshow(phi0.T, origin='lower', cmap='RdBu', vmin=0, vmax=1)
axes[1].set_title('Initial φ (oil=1, water=0)')
plt.colorbar(im1, ax=axes[1], shrink=0.5)

for ax in axes:
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

In [ ]:
# Run simulation
f_final, phi_final, Gamma_final, total_steps = run(
    step, f0, phi0, Gamma0, interior, p,
    chunk_size=500, n_chunks=60,
)

In [ ]:
# Summary statistics
rho_final, ux_final, uy_final = compute_macros(f_final)
vel_mag = jnp.sqrt(ux_final**2 + uy_final**2)
n_water = ((phi_final < 0.5).astype(jnp.float64) * interior).sum()
iface = (phi_final > 0.05) & (phi_final < 0.95)
gamma_mean = float(jnp.where(iface, Gamma_final, 0).sum() / jnp.maximum(iface.sum(), 1))

print(f"FLOW-FOCUSING — {total_steps} STEPS")
print(f"Max velocity: {float(vel_mag.max()):.4e} lu")
print(f"Water pixels: {float(n_water):.0f}")
print(f"phi range:    [{float(phi_final.min()):.6f}, {float(phi_final.max()):.6f}]")
print(f"rho range:    [{float(rho_final.min()):.6f}, {float(rho_final.max()):.6f}]")
print(f"Γ range:      [{float(Gamma_final.min()):.4f}, {float(Gamma_final.max()):.4f}]   ⟨Γ⟩_iface={gamma_mean:.3f}")

In [ ]:
# Final field plots: φ, ρ, u_y, |u|, Γ
plot_fields(f_final, phi_final, Gamma=Gamma_final)